In [5]:
import os
import certifi
import pymongo 
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Getting the CA certificate for secure connection
ca = certifi.where()

# MongoDB connection Details
MONGO_DB_URL = os.getenv("MONGO_URI")
DATABASE_NAME = "Visibility"
DATASET_DIR = r"D:\climate visibility project\notebooks"

# Establishing a connection to MongoDB

try:
    client = pymongo.MongoClient(MONGO_DB_URL, tlsCAFile=ca)
    database = client[DATABASE_NAME]
    print("Connected to MongoDB successfully!")
except Exception as e:
    raise Exception(f"Failed to connect to MongoDB: {e}")

# Function to load CSV files into MongoDB
def upload_files_to_mongodb(dataset_dir):
    for file in os.listdir(dataset_dir):
        if file.endswith('.csv'):
            collection_name = file.split('.')[0] # Use csv file name as collection name
            collection = database[collection_name]
            file_path = os.path.join(dataset_dir, file)
            print(f"processing file: {file_path}")
            df = pd.read_csv(file_path)
            # Remove extra spaces from column names
            df.columns = df.columns.str.strip()
            #convert all values to string 
            df = df.astype(str)
            #convert dataframes to list of dictionaries
            data = df.to_dict(orient='records')
            # Insert into MongoDB
            if data:
                collection.delete_many({})  # Clear existing data in the collection
                collection.insert_many(data)
                print(f"Uploaded {len(data)} records to collection: {collection_name}")
            else:
                print(f"No data found in file: {file_path}")


if __name__ == "__main__":
    upload_files_to_mongodb(DATASET_DIR)


Connected to MongoDB successfully!
processing file: D:\climate visibility project\notebooks\data.csv
Uploaded 75083 records to collection: data
